# Benchmarking recent EEG foundation models on iSLEEPSOur earlier experiments compared our own implementations against a classicalfeature + boosting pipeline, and every deep model lost. The fair objection is:*you did not test the architectures the field actually uses in 2024-2026.*This notebook answers that.Eight published architectures, **six of them from 2023-2025**, all trained and evaluatedunder one identical protocol:| model | reference | venue ||---|---|---|| **CBraMod** | Wang et al. 2025 | ICLR 2025 || **EEGPT** | Wang et al. 2024 | NeurIPS 2024 || **LaBraM** | Jiang et al. 2024 | ICLR 2024 || **BIOT** | Yang et al. 2023 | NeurIPS 2023 || **EEGConformer** | Song et al. 2023 | IEEE TNSRE || **SPARCNet** | Jing et al. 2023 | Neurology || U-Sleep | Perslev et al. 2021 | npj Digital Medicine || AttnSleep | Eldele et al. 2021 | IEEE TNSRE |Same folds, same class-balanced loss, same optimiser and schedule, same epoch budget,same HMM decoding. Only the architecture changes.### Scope limit, stated up front`braindecode` provides these **architectures**; we train them from scratch on iSLEEPS.This does **not** test the authors' **pretrained weights** — a CBraMod or LaBraMpretrained on thousands of hours of EEG and then fine-tuned here could behavedifferently. That is a separate experiment and a real limitation of this comparison.What this *does* test is whether these designs win on a 99-patient clinical cohort wheneveryone starts equal.

## Setup

In [ ]:
import os, glob, json, timeimport numpy as npimport torch, torch.nn as nnfrom torch.utils.data import Dataset, DataLoaderfrom sklearn.metrics import accuracy_score, f1_score, cohen_kappa_scoreDATA = "/kaggle/input/isleeps-processed7/processed7"if not os.path.isdir(DATA):    for alt in glob.glob("/kaggle/input/*/processed7") + ["data/processed7"]:        if os.path.isdir(alt): DATA = alt; breakFS, T_LEN = 100, 3000                    # 30 s epochs @ 100 HzCLASS_NAMES = ["W","N1","N2","N3","R"]DUPLICATE   = {28}                       # SN28 EDF payload == SN15# --- runtime budget: raise for the full protocol -----------------------------N_SUBJECTS = 30                          # None = all 99N_FOLDS    = 3                           # paper uses 10EPOCHS     = 8BATCH      = 64DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"torch.manual_seed(42); np.random.seed(42)print("data:", DATA, "| device:", DEVICE, "| braindecode", __import__("braindecode").__version__)

In [ ]:
paths = sorted(glob.glob(os.path.join(DATA, "SN*.npz")))sids  = [int(os.path.basename(p)[2:-4]) for p in paths]sids  = [s for s in sids if s not in DUPLICATE]if N_SUBJECTS: sids = sids[:N_SUBJECTS]X, Y = {}, {}for s in sids:    d = np.load(os.path.join(DATA, f"SN{s}.npz"), allow_pickle=True)    X[s] = d["x"].astype(np.float32); Y[s] = d["y"].astype(np.int64)cnt = np.bincount(np.concatenate(list(Y.values())), minlength=5)print(f"{len(sids)} subjects | {cnt.sum():,} epochs | channels {list(d['channels'])}")for c,n in zip(CLASS_NAMES,cnt): print(f"  {c:3s} {n:6,d} {100*n/cnt.sum():5.1f}%")

In [ ]:
def make_folds(subjects, n_splits, seed=42):    rng = np.random.RandomState(seed); sh=list(subjects); rng.shuffle(sh)    g=[sh[i::n_splits] for i in range(n_splits)]    return [(sorted(x for x in sh if x not in g[k]), sorted(g[k])) for k in range(n_splits)]FOLDS = make_folds(sids, N_FOLDS)for i,(tr,te) in enumerate(FOLDS): print(f"fold {i}: train {len(tr)} | test {len(te)}")

In [ ]:
# ---------- shared evaluation machinery: identical for every model ----------def fit_hmm(seqs):    A=np.ones((5,5)); pi=np.ones(5)    for y in seqs:        pi[y[0]]+=1        for a,b in zip(y[:-1],y[1:]): A[a,b]+=1    A/=A.sum(1,keepdims=True); pi/=pi.sum()    return np.log(A+1e-12), np.log(pi+1e-12)def viterbi(lp,lA,lpi):    T=lp.shape[0]; dp=np.zeros((T,5)); bp=np.zeros((T,5),int); dp[0]=lpi+lp[0]    for t in range(1,T):        sc=dp[t-1][:,None]+lA; bp[t]=sc.argmax(0); dp[t]=sc.max(0)+lp[t]    p=np.zeros(T,int); p[-1]=dp[-1].argmax()    for t in range(T-2,-1,-1): p[t]=bp[t+1,p[t+1]]    return pdef metrics(y,p):    return dict(acc=accuracy_score(y,p), mf1=f1_score(y,p,average="macro",zero_division=0),                kappa=cohen_kappa_score(y,p),                pcf=f1_score(y,p,average=None,labels=range(5),zero_division=0))class EpochDS(Dataset):    def __init__(self, subjects, chans):        self.items=[(s,i) for s in subjects for i in range(len(Y[s]))]; self.ch=chans    def __len__(self): return len(self.items)    def __getitem__(self,i):        s,j=self.items[i]; x=X[s][j][self.ch]        x=(x-x.mean(1,keepdims=True))/(x.std(1,keepdims=True)+1e-6)        return torch.from_numpy(x), int(Y[s][j])def class_weights(subs):    c=np.bincount(np.concatenate([Y[s] for s in subs]),minlength=5)    return torch.tensor(c.sum()/(5*np.maximum(c,1)),dtype=torch.float32,device=DEVICE)print("harness ready")

## Training harnessOne function for every model, so the comparison is fair by construction. LaBraM needsits channel names at forward time, which the wrapper handles.

In [ ]:
def run_model(name, build_fn, chans, ch_names=None, epochs=EPOCHS, batch=BATCH):    yt_all, yraw, yhmm = [], [], []    t0, n_par = time.time(), None    fwd = (lambda m,x: m(x, ch_names=ch_names)) if ch_names else (lambda m,x: m(x))    for fi,(tr,te) in enumerate(FOLDS):        model = build_fn().to(DEVICE)        if n_par is None: n_par = sum(p.numel() for p in model.parameters())        opt   = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)        crit  = nn.CrossEntropyLoss(weight=class_weights(tr))        # num_workers=0 on purpose: the whole dataset is already resident in RAM, so        # worker processes would fork a copy of it per worker for no gain (and break        # on spawn-based platforms).        dl    = DataLoader(EpochDS(tr,chans), batch_size=batch, shuffle=True,                           num_workers=0, drop_last=True, pin_memory=True)        for ep in range(epochs):            model.train(); run=0.0            for xb,yb in dl:                xb,yb=xb.to(DEVICE,non_blocking=True),yb.to(DEVICE,non_blocking=True)                opt.zero_grad(); loss=crit(fwd(model,xb),yb)                loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),5.0)                opt.step(); run+=loss.item()            sched.step()            print(f"    [{name}] fold{fi} ep{ep+1}/{epochs} loss {run/len(dl):.4f}", flush=True)        lA,lpi = fit_hmm([Y[s] for s in tr]); model.eval()        with torch.no_grad():            for s in te:                x=X[s][:,chans]                x=(x-x.mean(2,keepdims=True))/(x.std(2,keepdims=True)+1e-6)                pr=[]                for i in range(0,len(x),128):                    pr.append(torch.softmax(fwd(model,torch.from_numpy(x[i:i+128]).to(DEVICE)),-1).cpu().numpy())                pr=np.concatenate(pr)                yt_all.append(Y[s]); yraw.append(pr.argmax(1))                yhmm.append(viterbi(np.log(pr+1e-12),lA,lpi))        del model; torch.cuda.empty_cache()    yt=np.concatenate(yt_all)    r,h = metrics(yt,np.concatenate(yraw)), metrics(yt,np.concatenate(yhmm))    mins=(time.time()-t0)/60    print(f"  == {name}: raw {r['acc']:.4f}/{r['mf1']:.4f} | +HMM {h['acc']:.4f}/{h['mf1']:.4f}/"          f"{h['kappa']:.4f} | {n_par/1e6:.2f}M | {mins:.1f} min\n", flush=True)    return dict(name=name, params=n_par, minutes=mins, raw=r, hmm=h)

## The model zooChannel counts follow each paper's design: AttnSleep is single-channel, LaBraM maps ontothe standard 10-20 montage so it gets the four EEG derivations, the rest take all seven.

In [ ]:
from braindecode import models as MALL, EEG1, EEG4 = list(range(7)), [0], [0,1,2,3]LABRAM_CH = ["C4","C3","O2","O1"]     # C4:M1->C4, C3:M2->C3, O2:M1->O2, O1:M2->O1ZOO = [  ("CBraMod (ICLR'25)",   lambda: M.CBraMod(n_chans=7,n_outputs=5,n_times=T_LEN,sfreq=FS), ALL,  None),  ("EEGPT (NeurIPS'24)",  lambda: M.EEGPT(n_chans=7,n_outputs=5,n_times=T_LEN,sfreq=FS),   ALL,  None),  ("LaBraM (ICLR'24)",    lambda: M.Labram(n_chans=4,n_outputs=5,n_times=T_LEN,sfreq=FS),  EEG4, LABRAM_CH),  ("BIOT (NeurIPS'23)",   lambda: M.BIOT(n_chans=7,n_outputs=5,n_times=T_LEN,sfreq=FS),    ALL,  None),  ("EEGConformer ('23)",  lambda: M.EEGConformer(n_chans=7,n_outputs=5,n_times=T_LEN,sfreq=FS), ALL, None),  ("SPARCNet ('23)",      lambda: M.SPARCNet(n_chans=7,n_outputs=5,n_times=T_LEN,sfreq=FS),ALL,  None),  ("U-Sleep ('21)",       lambda: M.USleep(n_chans=7,sfreq=FS,n_outputs=5,depth=10,                                           input_window_seconds=30),                        ALL,  None),  ("AttnSleep ('21)",     lambda: M.AttnSleep(n_chans=1,n_outputs=5,n_times=T_LEN,sfreq=FS),EEG1, None),]for n,b,c,cn in ZOO:    m=b(); print(f"  {n:22s} {len(c)} ch  {sum(p.numel() for p in m.parameters())/1e6:7.2f}M"); del m

## Run

In [ ]:
RESULTS=[]for name, build, chans, chn in ZOO:    print(f"--- {name} ---", flush=True)    try:        RESULTS.append(run_model(name, build, chans, ch_names=chn))    except RuntimeError as e:        print(f"  {name} OOM/RUNTIME: {str(e)[:120]}\n  -> retrying at batch 16\n", flush=True)        try: RESULTS.append(run_model(name, build, chans, ch_names=chn, batch=16))        except Exception as e2: print(f"  {name} FAILED: {type(e2).__name__}: {str(e2)[:120]}\n")    except Exception as e:        print(f"  {name} FAILED: {type(e).__name__}: {str(e)[:120]}\n", flush=True)print(f"benchmark complete: {len(RESULTS)}/{len(ZOO)} models")

## Results

In [ ]:
OURS = ("Classical features + boosting + HMM (ours)", 0.7464, 0.6753, 0.6415)PUB  = ("LSTM, dataset paper (N=100)",                0.7470, 0.6768, 0.6400)print(f"{'model':26s} {'params':>9s} {'acc':>8s} {'macroF1':>9s} {'kappa':>8s} {'min':>6s}")print("-"*70)for r in sorted(RESULTS,key=lambda z:-z["hmm"]["mf1"]):    print(f"{r['name']:26s} {r['params']/1e6:8.2f}M {r['hmm']['acc']:8.4f} "          f"{r['hmm']['mf1']:9.4f} {r['hmm']['kappa']:8.4f} {r['minutes']:6.1f}")print("-"*70)for nm,a,f,k in (OURS,PUB): print(f"{nm:26s} {'--':>9s} {a:8.4f} {f:9.4f} {k:8.4f}")print(f"\n{'N1 F1 (the limiting class)':26s}")for r in sorted(RESULTS,key=lambda z:-z["hmm"]["pcf"][1]):    print(f"  {r['name']:24s} {r['hmm']['pcf'][1]:.3f}")print(f"  {'ours':24s} 0.315")json.dump([{"name":r["name"],"params":int(r["params"]),"minutes":r["minutes"],            "hmm":{k:(v.tolist() if hasattr(v,'tolist') else v) for k,v in r["hmm"].items()},            "raw":{k:(v.tolist() if hasattr(v,'tolist') else v) for k,v in r["raw"].items()}}           for r in RESULTS], open("benchmark_results.json","w"), indent=2)print("\nsaved -> benchmark_results.json")

## How to read this**The comparison is fair within the deep models** — shared folds, loss, schedule, epochbudget and decoding. Our classical row and the dataset paper's row are context only:they were produced under their own conditions, and the dataset paper additionally usedall 100 recordings including the SN28 duplicate.**Watch N1.** Roughly 10% of epochs but a fifth of macro-F1, and the class every methodon this cohort struggles with.**Three caveats worth keeping in mind.**1. These are **architectures trained from scratch**, not the authors' pretrained   checkpoints. Foundation models are designed to be pretrained on large corpora and   fine-tuned; denying them that is denying their main advantage. Treat a poor score   here as *"this design does not win from scratch on 99 patients"*, not as   *"foundation models do not work"*.2. Hyperparameters are shared, not tuned per model. That is what makes the comparison   controlled, but each architecture's own paper would tune it further.3. The config at the top uses a subset of subjects and folds so this finishes in one   session. Set `N_SUBJECTS = None`, `N_FOLDS = 10` before quoting any number.If these land near or below the classical pipeline under an identical protocol, thatsupports the data-limited reading. If one clearly wins, that is a real result and itbelongs in the paper, whichever way it cuts.